In [1]:
import pandas as pd
import torch
import numpy as np


states = torch.load("states.pt")
legal_masks = torch.load("legal_masks.pt")
target = pd.read_csv("target.csv")

In [2]:
DEVICE = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

In [3]:
row = states.shape[0]

print(states.shape)
print(legal_masks.shape)

torch.Size([707542, 30, 8, 8])
torch.Size([707542, 4672])


In [4]:
import sys
sys.path.append('..')

In [5]:
policy = torch.tensor(target.policy.values).float()
value = torch.tensor(target.value.values).float()


In [6]:
# Train test split
TRAIN_SIZE = int(0.9 * len(states))
assert len(states) == len(policy) == len(value) == len(legal_masks) 

train_states, test_states = states[:TRAIN_SIZE],      states[TRAIN_SIZE:]
train_policy, test_policy = policy[:TRAIN_SIZE],       policy[TRAIN_SIZE:]
train_value,  test_value  = value[:TRAIN_SIZE],        value[TRAIN_SIZE:]
train_masks,  test_masks  = legal_masks[:TRAIN_SIZE],  legal_masks[TRAIN_SIZE:]

assert len(train_states) == len(train_policy) == len(train_value) == len(train_masks)
assert len(test_states)  == len(test_policy)  == len(test_value)  == len(test_masks)

In [7]:

from core import factory

network = factory.build_network("chess")

In [8]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss, MSELoss

optimizer = Adam(network.parameters(), lr=3e-4, fused=True)
value_loss_fn = MSELoss()
policy_loss_fn = CrossEntropyLoss()

In [9]:
from core.network import PolicyValueNetwork
from torch import optim
import time
from core.network import PolicyValueNetwork
from torch import optim
import time
import torch
import numpy as np
from torch.optim import lr_scheduler


def evaluate(network, test_states, test_policy, test_value, test_masks,
             policy_loss_fn, value_loss_fn, batch_size=256, eval_samples=4096):
    network.eval()
    total_policy_loss, total_value_loss, n_batches = 0.0, 0.0, 0

    n = min(len(test_states), eval_samples)

    with torch.no_grad():
        for i in range(0, n, batch_size):
            batch_states = test_states[i:i+batch_size]
            batch_policy = test_policy[i:i+batch_size]
            batch_value  = test_value[i:i+batch_size]
            batch_mask   = test_masks[i:i+batch_size]

            policy_head, value_head = network(batch_states)
            policy_head = policy_head.masked_fill(~batch_mask.to(policy_head.device), float("-inf"))

            total_policy_loss += policy_loss_fn(policy_head, batch_policy).item()
            total_value_loss  += value_loss_fn(value_head, batch_value).item()
            n_batches += 1

    network.train()

    return total_policy_loss / n_batches, total_value_loss / n_batches


def train(network: PolicyValueNetwork,
          optimizer: optim.Optimizer,
          train_states: torch.Tensor,
          train_policy: torch.Tensor,
          train_value: torch.Tensor,
          train_masks: torch.Tensor, 
          policy_loss_fn,
          value_loss_fn,
          test_states: torch.Tensor | None = None,
          test_policy: torch.Tensor | None = None,
          test_value: torch.Tensor | None = None,
          test_masks: torch.Tensor | None = None,
          batch_size: int = 256,
          c: int = 1,
          num_iter: int | None = None,
          duration_hour: float | None = None,
          eval_every: int = 100,
          seed: int = 42):

    if num_iter is None and duration_hour is None:
        raise ValueError("Must specify at least one of num_iter or duration_hour")

    start = time.time()
    rng = np.random.default_rng(seed=seed)
    step = 0

    warmup_steps = 200
    warmup = lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)
    cosine = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_iter - warmup_steps if num_iter else 9999)
    scheduler = lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

    train_policy = train_policy.to(device=DEVICE)
    train_value  = train_value.unsqueeze(-1).to(device=DEVICE)
    # train_masks stays on CPU; only the batch slice moves to device below.

    has_test = test_states is not None and test_policy is not None and test_value is not None
    if has_test:
        test_states = test_states.to(device=DEVICE)
        test_policy = test_policy.to(device=DEVICE)
        test_value  = test_value.unsqueeze(-1).to(device=DEVICE)

    while True:
        if num_iter is not None and step >= num_iter:
            break
        if duration_hour is not None and time.time() - start >= duration_hour * 3600:
            break

        batch_idx = rng.choice(len(train_states), batch_size, replace=False)
        batch_train_states = train_states[batch_idx]
        batch_train_policy = train_policy[batch_idx]
        batch_train_value  = train_value[batch_idx]
        batch_mask         = train_masks[batch_idx].to(DEVICE)

        optimizer.zero_grad()
        policy_head, value_head = network(batch_train_states)
        policy_head = policy_head.masked_fill(~batch_mask, float("-inf"))

        policy_loss = policy_loss_fn(policy_head, batch_train_policy)
        value_loss  = c * value_loss_fn(value_head, batch_train_value)
        loss = policy_loss + value_loss
        loss.backward()

        optimizer.step()
        scheduler.step()

        if step % 10 == 0:
            elapsed = time.time() - start
            print(f"[{step}] loss={loss.item():.4f} | policy={policy_loss.item():.4f} | value={(value_loss.item()):.4f} | | lr: {scheduler.get_last_lr()[0]:.8f} | {elapsed:.0f}s")

        if has_test and step % eval_every == 0 and step > 0:
            val_policy_loss, val_value_loss = evaluate(
                network, test_states, test_policy, test_value, test_masks,
                policy_loss_fn, value_loss_fn
            )
            print(f"    [eval @ {step}] val_policy={val_policy_loss:.4f} | val_value={val_value_loss:.4f}")

        step += 1

In [ ]:
train(
    duration_hour=1,
    network=network,
    optimizer=optimizer,
    train_states=train_states,
    train_policy=train_policy,
    train_value=train_value,
    train_masks=train_masks,
    test_states=test_states,
    test_policy=test_policy,
    test_value=test_value,
    test_masks=test_masks,
    c=6,
    policy_loss_fn=policy_loss_fn,
    value_loss_fn=value_loss_fn,
    batch_size=64,
    seed=42
)

[0] loss=10.1131 | policy=5.0220 | value=5.0911 | | lr: 0.00000448 | 1s
[10] loss=8.2860 | policy=4.1854 | value=4.1006 | | lr: 0.00001934 | 6s
[20] loss=8.8495 | policy=4.5557 | value=4.2938 | | lr: 0.00003419 | 15s
[30] loss=8.5205 | policy=3.9665 | value=4.5540 | | lr: 0.00004903 | 23s
[40] loss=7.4958 | policy=4.1375 | value=3.3583 | | lr: 0.00006389 | 31s
[50] loss=7.5408 | policy=4.0597 | value=3.4811 | | lr: 0.00007874 | 39s
[60] loss=10.3164 | policy=4.0578 | value=6.2586 | | lr: 0.00009358 | 47s
[70] loss=8.1339 | policy=3.8654 | value=4.2685 | | lr: 0.00010844 | 55s
[80] loss=6.8294 | policy=3.6763 | value=3.1530 | | lr: 0.00012329 | 63s
[90] loss=7.5078 | policy=3.5588 | value=3.9490 | | lr: 0.00013814 | 71s
[100] loss=7.5198 | policy=3.5351 | value=3.9846 | | lr: 0.00015299 | 79s
    [eval @ 100] val_policy=4.0862 | val_value=1.0615
[110] loss=7.3652 | policy=3.6237 | value=3.7415 | | lr: 0.00016784 | 97s
[120] loss=6.9024 | policy=3.4625 | value=3.4400 | | lr: 0.00018269 |

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:243: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[200] loss=5.9446 | policy=3.2906 | value=2.6539 | | lr: 0.00030000 | 171s
    [eval @ 200] val_policy=4.6755 | val_value=0.6102
[210] loss=6.9875 | policy=3.3094 | value=3.6782 | | lr: 0.00030000 | 191s
[220] loss=8.0275 | policy=3.2317 | value=4.7958 | | lr: 0.00030000 | 199s
[230] loss=7.2326 | policy=3.2553 | value=3.9773 | | lr: 0.00029999 | 207s
[240] loss=7.3120 | policy=3.4260 | value=3.8860 | | lr: 0.00029999 | 216s
[250] loss=7.4360 | policy=3.5285 | value=3.9075 | | lr: 0.00029998 | 224s
[260] loss=6.8046 | policy=3.2642 | value=3.5404 | | lr: 0.00029997 | 231s
[270] loss=6.6206 | policy=3.6421 | value=2.9785 | | lr: 0.00029996 | 239s
[280] loss=7.1750 | policy=3.5189 | value=3.6561 | | lr: 0.00029995 | 247s
[290] loss=6.0322 | policy=3.1253 | value=2.9068 | | lr: 0.00029994 | 256s
[300] loss=7.1132 | policy=3.4020 | value=3.7113 | | lr: 0.00029992 | 265s
    [eval @ 300] val_policy=3.2771 | val_value=0.5573
[310] loss=5.5844 | policy=3.3053 | value=2.2791 | | lr: 0.00029991